# Reference Tracking과 Optimal Control


## 학습 목표

<!-- exercise-guidance:control-2/reference-optimal-control/objective -->
안정한 제어기에서도 목표값 오차가 남는 이유를 확인하고, 추종 성능과 제어입력 비용을 비교합니다.
<!-- /exercise-guidance:control-2/reference-optimal-control/objective -->


## 목표값을 따라가도록 Reference 이득 정하기

<!-- exercise-guidance:control-2/reference-optimal-control/activity-1 -->
### 시작 조건

A=[[0,1],[-2,-3]], B=[[0],[1]], C=[1,0], K=[6,3], r=1, x(0)=0입니다.

### 진행 방법

1. N=1과 N=8의 응답과 Steady State 값을 비교하세요.
2. 0=(A−BK)x+B*N*r에서 Steady State y=r을 만족하는 N을 유도하세요.

### 남길 결과

- 두 Step 응답과 Steady State 값, N을 구한 식.
<!-- /exercise-guidance:control-2/reference-optimal-control/activity-1 -->


In [ ]:
import numpy as np
from scipy.signal import place_poles,StateSpace,lsim
from scipy.linalg import solve_continuous_are
import matplotlib.pyplot as plt
A=np.array([[0.,1.],[-2.,-3.]])
B=np.array([[0.],[1.]]);C=np.array([[1.,0.]])
K=place_poles(A,B,[-2.,-4.]).gain_matrix
N=float((-1/(C@np.linalg.solve(A-B@K,B))).item())
print('K:',K,'N:',N);assert np.isclose(N,8)
t=np.linspace(0,5,1001)
for gain in [1.,N]:
    _,y,_=lsim(StateSpace(A-B@K,B*gain,C,0),np.ones(len(t)),t)
    plt.plot(t,y,label=f'N={gain:g}')
plt.xlabel('Time [s]');plt.ylabel('Position');plt.legend();plt.grid()



## 결과 기록·자가 점검

<!-- exercise-guidance:control-2/reference-optimal-control/activity-1-record -->
**N:**

**Steady State 값:**

**Closed-loop Pole:**

### 확인할 점

- N=1의 Steady State 값은 1/8, N=8은 1인가?
- N을 바꾸어도 Closed-loop Pole은 같은가?
<!-- /exercise-guidance:control-2/reference-optimal-control/activity-1-record -->


## 출력 오차와 입력 비용의 절충

<!-- exercise-guidance:control-2/reference-optimal-control/activity-2 -->
### 시작 조건

Q=diag([q,1]), q=1,25이며 R=[1]입니다. x(0)=[1,0], r=0입니다.

### 진행 방법

1. 각 Q의 CARE를 풀고 K=R⁻¹BᵀP를 계산하세요.
2. 같은 시간축의 위치와 제어입력을 그리세요.
3. 위치 오차 제곱의 적분과 입력 제곱의 적분을 별도로 비교하세요.

### 남길 결과

- 위치·입력 그래프와 두 적분값의 비교표.
<!-- /exercise-guidance:control-2/reference-optimal-control/activity-2 -->


In [ ]:
from scipy.linalg import solve_continuous_are
for q in [1.,25.]:
    Q=np.diag([q,1.]);P=solve_continuous_are(A,B,Q,[[1.]])
    Kq=B.T@P
    _,x,_=lsim(StateSpace(A-B@Kq,np.zeros((2,1)),np.eye(2),np.zeros((2,1))),np.zeros(len(t)),t,X0=[1,0])
    u=-(x@Kq.T).ravel()
    print(q,Kq,np.trapezoid(x[:,0]**2,t),np.trapezoid(u**2,t))
    plt.plot(t,x[:,0],label=f'q={q:g}')
plt.xlabel('Time [s]');plt.ylabel('Position');plt.legend();plt.grid()
# TODO: 제어입력 그래프와 극점 확인을 추가하세요.


## 결과 기록·자가 점검

<!-- exercise-guidance:control-2/reference-optimal-control/activity-2-record -->
**q:**

**K:**

**위치 오차 적분:**

**입력 비용:**

### 확인할 점

- 두 Q가 모두 양의 정부호이고 R>0인가?
- A−BK의 모든 Pole 실수부가 음수인가?
<!-- /exercise-guidance:control-2/reference-optimal-control/activity-2-record -->


## 모델 오차가 있을 때의 목표 추종

<!-- exercise-guidance:control-2/reference-optimal-control/activity-3 -->
### 시작 조건

설계는 명목 B로 하고, 실제 입력 이득만 B_actual=g*B, g=.8,1,1.2로 바꿉니다. K=[6,3], N=8, r=1입니다.

### 진행 방법

1. 각 실제 모델의 Pole과 Step 최종값을 구하세요.
2. 명목 모델에서 계산한 N이 모든 실제 모델에서도 목표값을 정확히 맞추는지 확인하세요.
3. Reference 이득 조정과 오차 적분 Feedback의 역할을 구분하세요.

### 남길 결과

- 세 응답 곡선과 최종값·오차 표.
<!-- /exercise-guidance:control-2/reference-optimal-control/activity-3 -->


In [ ]:
K=np.array([[6.,3.]])
for g in [.8,1.,1.2]:
    Ba=g*B;Acl=A-Ba@K
    steady=float((-C@np.linalg.solve(Acl,Ba*8)).item())
    _,y,_=lsim(StateSpace(Acl,Ba*8,C,0),np.ones(len(t)),t)
    print(g,np.linalg.eigvals(Acl),steady);plt.plot(t,y,label=str(g))
plt.xlabel('Time [s]');plt.ylabel('Position');plt.legend();plt.grid()
# TODO: 오차가 남는 이유와 적분 Feedback이 추가로 해결할 역할을 설명하세요.


## 결과 기록·자가 점검

<!-- exercise-guidance:control-2/reference-optimal-control/activity-3-record -->
**입력 이득:**

**Pole:**

**최종값:**

**추종 오차:**

### 확인할 점

- Feedback 항과 Reference 항 모두 실제 B_actual에 곱해졌는가?
- 최종값을 계산하기 전에 안정성을 확인했는가?
<!-- /exercise-guidance:control-2/reference-optimal-control/activity-3-record -->


## 공식 예제·데이터

<!-- exercise-guidance:control-2/reference-optimal-control/resources -->
- [SciPy · Pole placement](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.place_poles.html)
- [SciPy · Continuous-time CARE](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.solve_continuous_are.html)
<!-- /exercise-guidance:control-2/reference-optimal-control/resources -->
